# TCGA Classifiers


In [ ]:

import pandas as pd

In [ ]:
# Save train and test data to CSV files
X_train = pd.read_csv('../DATA/tcga/X_train.csv')
X_test = pd.read_csv('../DATA/tcga/X_test.csv')
y_train = pd.read_csv('../DATA/tcga/y_train.csv')
y_test = pd.read_csv('../DATA/tcga/y_test.csv')

## Selection of target genes

An important part of our analysis is the selection of genes that are connected to the mutated/non-mutated conditions of TP53 to make our prediction.

We identified a list of TP53 target genes—genes whose expression is directly regulated by the TP53 protein. To make predictions, we aim to select a subset of these target genes.

For selection, we rank the target genes based on the absolute value of their Spearman correlation coefficients: for each gene, we calculate the Spearman correlation coefficient with the target variable. Spearman correlation measures the relationship between two variables, providing insight into how changes in one variable are related to changes in another. We only retain genes where the p-value of the correlation is less than or equal to 0.001. This threshold ensures that we focus on genes with statistically significant correlations, reducing the risk of selecting genes with weak or insignificant relationships to the target variable. Finally, we select the top 100 genes with the highest absolute Spearman correlation coefficients. These genes are most strongly associated with the target variable.

Additionally, we explore another approach where we rank all the genes in our dataset according to their Spearman correlation coefficient (not limited to just TP53 target genes) and select a subset of these for classifier construction. We evaluate the performance of both methods to determine which yields the best results.

Our analysis begins by selecting a subset of TP53 target genes with the highest absolute Spearman correlation coefficients.

In [ ]:
target_genes = [
    'CDKN1A', 'ABCA12', 'NTPCR', 'PGPEP1', 'RNF19B', 'LCE1E', 'EPN3', 'SCN4B', 'ARVCF', 'FCHO2',
    'PANK2', 'TMEM8B', 'RRM2B', 'ANKRA2', 'ORAI3', 'PLCL2', 'SAC3D1', 'LIMK2', 'FBXO32', 'SCRIB',
    'BHLHE40', 'FCHSD2', 'PAQR7', 'TP53', 'MDM2', 'CCNG1', 'PRKAB1', 'PMAIP1', 'SYTL1', 'LRP1',
    'FHL2', 'SEMA3B', 'BMP7', 'FLRT2', 'PCBP4', 'TP53I11', 'SUSD6', 'CYFIP2', 'PTP4A1', 'PRDM1',
    'TNFRSF10A', 'MCC', 'HES2', 'SLC25A45', 'BORCS7', 'GBE1', 'PERP', 'TRAK1', 'GDF15', 'DRAM1',
    'SESN2', 'RAP2B', 'TNFRSF10D', 'NUPR1', 'KCNN4', 'SLC44A5', 'BTBD10', 'GPC1', 'PLLP', 'TRIP6',
    'BTG2', 'FBXO22', 'SLC30A1', 'RRAD', 'TSPAN11', 'PARD6G', 'KLHDC7A', 'SLC4A11', 'BTG3', 'HES1',
    'POU3F1', 'TSGA10', 'DDB2', 'ISCU', 'SPATA18', 'ZNF219', 'VWCE', 'PHPT1', 'LMNA', 'SLC9A1',
    'C17orf89', 'HRAS', 'PPFIBP1', 'UNC5B', 'GADD45A', 'PHLDA3', 'TGFA', 'ZNF337', 'DDIT4', 'PIDD1',
    'MLF2', 'STAT3', 'CAPN2', 'HSD17B3', 'PPM1J', 'UQCC1', 'PLK3', 'SERPINB5', 'TLR3', 'ACTA2',
    'RAD51C', 'PML', 'MR1', 'STK17A', 'CASP6', 'ICOSLG', 'PPP4R3A', 'VDR', 'TIGAR', 'SERTAD1',
    'TM7SF3', 'EDN2', 'SERPINE1', 'PTPRE', 'MYO6', 'STX6', 'CATSPERG', 'IGFBP7', 'PTAFR', 'YPEL3',
    'RPS27L', 'TRAF4', 'TMEM68', 'ALOX5', 'TNFAIP8', 'PVRL4', 'NEFL', 'TP73', 'CAV1', 'IL1B',
    'RALGDS', 'ZNF195', 'TNFRSF10B', 'TRIM22', 'WDR63', 'ARHGEF3', 'TSKU', 'RETSAT', 'NKAIN4',
    'TRIM32', 'CCNK', 'ISYNA1', 'RBM38', 'ZNF385A', 'TRIAP1', 'CES2', 'ZNF561', 'CERS5', 'PCNA',
    'REV3L', 'PCLO', 'TRIM38', 'CFLAR', 'JAG1', 'RGL1', 'ZNF488', 'ZMAT3', 'CMBL', 'ZNF79',
    'DDR1', 'ACYP2', 'RNASE7', 'PDE4C', 'TRIM5', 'CGB7', 'KRT8', 'RGS20', 'BAX', 'FBXW7',
    'ASCC3', 'DHRS3', 'APAF1', 'SFN', 'PGAP1', 'TYMSOS', 'CHST14', 'KSR1', 'RHOC', 'PGF',
    'HSPA4L', 'ACER2', 'DUSP14', 'APOBEC3H', 'TNFRSF10C', 'PLCXD2', 'AKAP9', 'COBLL1', 'LACC1',
    'RPS19', 'POLH', 'KITLG', 'ANXA4', 'E2F7', 'BCL2L1', 'TRIML2', 'PLEKHF1', 'CCDC51', 'CPEB2',
    'LPXN', 'SARS', 'PPM1D', 'SLC12A4', 'APOBEC3C', 'EPS8L2', 'BCL6', 'VCAN', 'PLTP', 'CDH8',
    'CPSF4', 'LRPAP1', 'SCIN', 'SULF2', 'ATF3', 'ASTN2', 'FAM210B', 'BLCAP', 'ADCK3', 'PLXNB1',
    'DUSP11', 'DNAJB2', 'MFAP3L', 'SCN3B', 'XPC', 'BBC3', 'CD82', 'GLS2', 'C17orf82', 'AK3',
    'PLXNB2', 'GCC2', 'DOCK8', 'MKNK2', 'SDC4', 'AEN', 'CCDC90B', 'CDIP1', 'GPX1', 'COL7A1',
    'ALDH1A3', 'PRKAB2', 'METTL8', 'DUSP5', 'MON2', 'SDPR', 'BLOC1S2', 'DYRK3', 'CPE', 'GRHL3',
    'CPEB4', 'BBS2', 'PRKX', 'PPP1R3C', 'DUSP7', 'MRPL49', 'SMAD3', 'FAS', 'EDA2R', 'CSF1',
    'HHAT', 'CSNK1G1', 'BTG1', 'PRODH', 'STEAP3', 'EBI3', 'MYBPHL', 'SNX2', 'GPR87', 'EPHA2',
    'DCP1B', 'IGDCC4', 'DGKA', 'CEL', 'PTPRU', 'ABHD4', 'EFNB1', 'MYLK', 'SOCS4', 'NINJ1',
    'FAM13C', 'ENC1', 'IKBIP', 'FAM49A', 'CLCA2', 'RGMA', 'ABTB2', 'EI24', 'MYOF', 'TAB3',
    'PLK2', 'FAM198B', 'FOSL1', 'LAPTM5', 'FAM84B', 'CLDN1', 'RGS16', 'ADGRG1', 'EML2', 'NFKBIA',
    'TCAIM', 'PSTPIP2', 'FAM212B', 'FUCA1', 'MAST4', 'GNAI1', 'CLP1', 'RND3', 'AIFM2', 'ENPP2',
    'NHLH2', 'TEP1', 'SESN1', 'FDXR', 'IER5', 'MICALL1', 'INPP1', 'CROT', 'RNF144B', 'AMOTL1',
    'ETV7', 'NLRP1', 'TET2', 'TP53I3', 'LIF', 'PADI4', 'NOTCH1', 'ITGA3', 'CYP4F3', 'S100A2',
    'AMZ2', 'FAM196A', 'NYNRIN', 'TEX9', 'TP53INP1', 'NADSYN1', 'PANK1', 'RABGGTA', 'KRT15',
    'DAPK1', 'SCN2A', 'ARC', 'FAM98C', 'P3H2', 'TMEM63B'
]

### TP53 Target Genes Approach

We first filter our dataset to only include known TP53 target genes:

In [ ]:
selected_dataset = X_train[target_genes]
selected_dataset.head()

From these target genes, we select the top 100 most significantly correlated with TP53 status using Spearman correlation (p ≤ 0.001):

In [ ]:
def select_genes_spearman(dataset):
    correlations = []
    p_values = []
    selected_genes = []
    for column in dataset.columns:
        corr, p_value = spearmanr(dataset[column], y_train)
        if p_value <= 0.001:
            correlations.append(corr)
            p_values.append(p_value)
            selected_genes.append(column)

    corr_results = pd.DataFrame({
        'Gene': selected_genes,
        'Spearman_Correlation': correlations,
        'P_Value': p_values
    })
    corr_results['Abs_Correlation'] = corr_results['Spearman_Correlation'].abs()
    corr_results = corr_results.sort_values(by='Abs_Correlation', ascending=False)

    selected_genes = corr_results.head(100)['Gene'].tolist()
    return selected_genes

In [ ]:
# selected genes among the target ones
selected_target_genes = select_genes_spearman(selected_dataset)
print(selected_target_genes)
print(len(selected_target_genes))

In [ ]:
X_train_target_genes = X_train[selected_target_genes]
X_train_target_genes

In [ ]:
X_test_target_genes = X_test[selected_target_genes]

For comparison, we also apply the Spearman correlation method to all genes to select the top 100 most correlated genes.

In [ ]:
# selected genes among all the ones present in the dataset
selected_genes = select_genes_spearman(X_train)
print(selected_genes)
print(len(selected_genes))

In [ ]:
X_train_subset = X_train[selected_genes]
X_test_subset = X_test[selected_genes]

## Models

### Binary classifier

Our initial approach is to construct a binary classifier aimed at predicting whether TP53 is mutated or not. The prediction model relies solely on the subset of genes we selected earlier — first using the subset of TP53 target genes and later using the second subset of all genes. We will then compare the performance of these two subsets to evaluate which provides the most accurate predictions.

Classifiers implemented:
* Random Forest: effective for high-dimensional data and captures complex interactions
* Logistic Regression: simple baseline model with good interpretability
* Support Vector Machine: works well with high-dimensional data
* K-Nearest Neighbors: might capture local patterns
* XGBoost: powerful gradient boosting method often top-performing
* Neural Network (MLP): can model complex non-linear relationships

Each classifier is implemented with grid search for hyperparameter tuning.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve
from sklearn.model_selection import GridSearchCV

In [ ]:
# Random Forest

def random_forest(X_train, y_train):

    param_grid_rf = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['log2', 'sqrt']
    }

    grid_search_rf = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_grid=param_grid_rf,
        cv=5,
        n_jobs=-1,
        verbose=2
    )

    grid_search_rf.fit(X_train, y_train)

    best_params_rf = grid_search_rf.best_params_
    print("Best Random Forest Parameters:", best_params_rf)


    best_rf = grid_search_rf.best_estimator_
    return best_rf

In [ ]:
# Logistic Regression

def logistic_regression(X_train, y_train):
 
    param_grid_lr = {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga']
    }

    grid_search_lr = GridSearchCV(
        estimator=LogisticRegression(random_state=42, max_iter=1000),
        param_grid=param_grid_lr,
        cv=5,
        n_jobs=-1,
        verbose=2
    )

    grid_search_lr.fit(X_train, y_train)

    best_params_lr = grid_search_lr.best_params_
    print("Best Logistic Regression Parameters:", best_params_lr)

    best_lr = grid_search_lr.best_estimator_
    return best_lr

In [ ]:
# Support Vector Machine

def svm(X_train, y_train):
   
    param_grid_svc = {
        'C': [1, 10],
        'kernel': ['rbf', 'poly'],
        'gamma': ['scale', 0.1, 1]
    }

    grid_search_svc = GridSearchCV(
        estimator=SVC(random_state=42),
        param_grid=param_grid_svc,
        cv=5,
        n_jobs=-1,
        verbose=2
    )

    grid_search_svc.fit(X_train, y_train)

    best_params_svc = grid_search_svc.best_params_
    print("Best SVM Parameters:", best_params_svc)

    best_svc = grid_search_svc.best_estimator_
    return best_svc

In [ ]:
# K-Nearest Neighbors

def knn(X_train, y_train):
  
    param_grid_knn = {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski']
    }

    grid_search_knn = GridSearchCV(
        estimator=KNeighborsClassifier(),
        param_grid=param_grid_knn,
        cv=5,
        n_jobs=-1,
        verbose=2
    )

    grid_search_knn.fit(X_train, y_train)

    best_params_knn = grid_search_knn.best_params_
    print("Best KNN Parameters:", best_params_knn)

    best_knn = grid_search_knn.best_estimator_
    return best_knn

In [ ]:
# XGBoost
def xgboost(X_train, y_train):
    param_grid_xgb_binary = {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }


    xgb = XGBClassifier(
        objective='binary:logistic',
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )


    grid_search_xgb = GridSearchCV(
        estimator=xgb,
        param_grid=param_grid_xgb_binary,
        cv=5,
        n_jobs=-1,
        verbose=2
    )

    grid_search_xgb.fit(X_train, y_train)

    print("Best XGBoost Parameters:", grid_search_xgb.best_params_)

    best_xgb = grid_search_xgb.best_estimator_
    return best_xgb

In [ ]:
# Neural Network (MLP)

def neural_netw(X_train, y_train):
    param_grid_nn = {
        'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 100)],
        'activation': ['relu', 'tanh'],
        'solver': ['adam', 'sgd'],
        'alpha': [0.0001, 0.001, 0.01], 
        'learning_rate': ['constant', 'adaptive']
    }

    grid_search_nn = GridSearchCV(
        estimator=MLPClassifier(max_iter=1000, random_state=42),
        param_grid=param_grid_nn,
        cv=5,
        n_jobs=-1,
        verbose=2
    )

    grid_search_nn.fit(X_train, y_train)

    best_params_nn = grid_search_nn.best_params_
    print("Best Neural Network Parameters:", best_params_nn)

    best_nn = grid_search_nn.best_estimator_
    return best_nn

Models are evaluated on the test set using accuracy, precision, recall, and F1 score.

In [ ]:
best_rf = random_forest(X_train_target_genes, y_train)
y_pred_rf = best_rf.predict(X_test_target_genes)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

In [ ]:
best_lr = logistic_regression(X_train_target_genes, y_train)
y_pred_lr = best_lr.predict(X_test_target_genes)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

In [ ]:
best_svc = svm(X_train_target_genes, y_train)
y_pred_svc = best_svc.predict(X_test_target_genes)

accuracy_svc = accuracy_score(y_test, y_pred_svc)
precision_svc = precision_score(y_test, y_pred_svc)
recall_svc = recall_score(y_test, y_pred_svc)
f1_svc = f1_score(y_test, y_pred_svc)

In [ ]:
best_knn = knn(X_train_target_genes, y_train)
y_pred_knn = best_knn.predict(X_test_target_genes)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

In [ ]:
best_xgb = xgboost(X_train_target_genes, y_train)
y_pred_xgb = best_xgb.predict(X_test_target_genes)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

In [ ]:
best_nn = neural_netw(X_train_target_genes, y_train)
y_pred_nn = best_nn.predict(X_test_target_genes)

accuracy_nn = accuracy_score(y_test, y_pred_nn)
precision_nn = precision_score(y_test, y_pred_nn)
recall_nn = recall_score(y_test, y_pred_nn)
f1_nn = f1_score(y_test, y_pred_nn)

In [ ]:
classifiers = [
    ("Random Forest", accuracy_rf, precision_rf, recall_rf, f1_rf),
    ("Logistic Regression", accuracy_lr, precision_lr, recall_lr, f1_lr),
    ("SVM", accuracy_svc, precision_svc, recall_svc, f1_svc),
    ("K-Nearest Neighbors", accuracy_knn, precision_knn, recall_knn, f1_knn),
    ("XGBoost", accuracy_xgb, precision_xgb, recall_xgb, f1_xgb),
    ("Neural Network (MLP)", accuracy_nn, precision_nn, recall_nn, f1_nn)
]

for clf_name, accuracy, precision, recall, f1 in classifiers:
    print(f"{clf_name} Performance:")
    print(f"  Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1 Score: {f1:.4f}\n")

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_svc = confusion_matrix(y_test, y_pred_svc)
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_nn = confusion_matrix(y_test, y_pred_nn)

In [ ]:
classifier_names = [
    'Random Forest', 'Logistic Regression', 'SVM',
    'K-Nearest Neighbors', 'XGBoost', 'Neural Network'
]

fig, axes = plt.subplots(2, 3, figsize=(10, 8))
fig.suptitle('Confusion Matrices', fontsize=16)

axes = axes.ravel()


for i, (cm, name) in enumerate(zip(
    [cm_rf, cm_lr, cm_svc, cm_knn, cm_xgb, cm_nn],
    classifier_names
)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

Overall the best F1 score is achieved by Random Forest with parameters {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}.

We now assess the performance of the same classifiers, but this time trained on a dataset that includes a subset of genes with the highest absolute Spearman correlations, without limiting the selection to target genes.

In [ ]:
best_rf = random_forest(X_train_subset, y_train)
y_pred_rf = best_rf.predict(X_test_subset)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

In [ ]:
best_lr = logistic_regression(X_train_subset, y_train)
y_pred_lr = best_lr.predict(X_test_subset)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

In [ ]:
best_svc = svm(X_train_subset, y_train)
y_pred_svc = best_svc.predict(X_test_subset)

accuracy_svc = accuracy_score(y_test, y_pred_svc)
precision_svc = precision_score(y_test, y_pred_svc)
recall_svc = recall_score(y_test, y_pred_svc)
f1_svc = f1_score(y_test, y_pred_svc)

In [ ]:
best_knn = knn(X_train_subset, y_train)
y_pred_knn = best_knn.predict(X_test_subset)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

In [ ]:
best_xgb = xgboost(X_train_subset, y_train)
y_pred_xgb = best_xgb.predict(X_test_subset)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

In [ ]:
best_nn = neural_netw(X_train_subset, y_train)
y_pred_nn = best_nn.predict(X_test_subset)

accuracy_nn = accuracy_score(y_test, y_pred_nn)
precision_nn = precision_score(y_test, y_pred_nn)
recall_nn = recall_score(y_test, y_pred_nn)
f1_nn = f1_score(y_test, y_pred_nn)

In [ ]:
classifiers = [
    ("Random Forest", accuracy_rf, precision_rf, recall_rf, f1_rf),
    ("Logistic Regression", accuracy_lr, precision_lr, recall_lr, f1_lr),
    ("SVM", accuracy_svc, precision_svc, recall_svc, f1_svc),
    ("K-Nearest Neighbors", accuracy_knn, precision_knn, recall_knn, f1_knn),
    ("XGBoost", accuracy_xgb, precision_xgb, recall_xgb, f1_xgb),
    ("Neural Network (MLP)", accuracy_nn, precision_nn, recall_nn, f1_nn)
]

for clf_name, accuracy, precision, recall, f1 in classifiers:
    print(f"{clf_name} Performance:")
    print(f"  Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1 Score: {f1:.4f}\n")

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_svc = confusion_matrix(y_test, y_pred_svc)
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_nn = confusion_matrix(y_test, y_pred_nn)

In [ ]:
# Classifier names for titles
classifier_names = [
    'Random Forest', 'Logistic Regression', 'SVM',
    'K-Nearest Neighbors', 'XGBoost', 'Neural Network'
]

# Create a figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(10, 8))
fig.suptitle('Confusion Matrices', fontsize=16)

# Flatten axes for easy iteration
axes = axes.ravel()

# Plot each confusion matrix
for i, (cm, name) in enumerate(zip(
    [cm_rf, cm_lr, cm_svc, cm_knn, cm_xgb, cm_nn],
    classifier_names
)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

The performance of the models trained on the top 100 genes—selected based on absolute Spearman correlation without restricting to TP53 target genes—is comparable to that of models trained solely on the TP53 target gene subset. This indicates that the subset of selected target genes is informative and effective for addressing the binary classification task.

The highest overall F1 score is obtained using a Random Forest model trained on the TP53 target gene subset with optimized hyperparameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}. Performance: Accuracy: 0.8569 | Precision: 0.7701 | Recall: 0.8306 | F1 Score: 0.7992

### Multi-class classifier
We repeat the same procedure, trying now to predict the specific mutation of TP53.

We use the same subsets of genes selected before to make our predictions.

In [ ]:
len(mutations['Variant_Type'].unique())

In [ ]:
y_train = y_train_initial
y_test = y_test_initial

In [ ]:
print(f"Training data shape: {X_train.shape}")
print(f"Training labels distribution:\n{y_train.value_counts()}")
print()
print(f"Testing data shape: {X_test.shape}")
print(f"Testing labels distribution:\n{y_test.value_counts()}")

In this multi-class classification task, the dataset is imbalanced, with some TP53 mutation types being much more common than others. To evaluate model performance fairly, we use average='weighted' for metrics like precision, recall, and F1 score.

This approach calculates the metric for each class and then averages them, weighted by the number of true instances per class. This ensures that more frequent classes have a greater impact on the final score, while still including performance on rare classes.

As before, we begin with the dataset using the subset of target genes as features.

In [ ]:
best_rf = random_forest(X_train_target_genes, y_train)
y_pred_rf = best_rf.predict(X_test_target_genes)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, average='weighted')
recall_rf = recall_score(y_test, y_pred_rf, average='weighted')
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

In [ ]:
best_lr = logistic_regression(X_train_target_genes, y_train)
y_pred_lr = best_lr.predict(X_test_target_genes)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

In [ ]:
best_svc = svm(X_train_target_genes, y_train)
y_pred_svc = best_svc.predict(X_test_target_genes)

accuracy_svc = accuracy_score(y_test, y_pred_svc)
precision_svc = precision_score(y_test, y_pred_svc, average='weighted')
recall_svc = recall_score(y_test, y_pred_svc, average='weighted')
f1_svc = f1_score(y_test, y_pred_svc, average='weighted')

In [ ]:
best_knn = knn(X_train_target_genes, y_train)
y_pred_knn = best_knn.predict(X_test_target_genes)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')

In [ ]:
from sklearn.preprocessing import LabelEncoder

# XGBoost is expecting numerical labels --> need to convert labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


best_xgb = xgboost(X_train_target_genes, y_train_encoded)
y_pred_xgb = best_xgb.predict(X_test_target_genes)

accuracy_xgb = accuracy_score(y_test_encoded, y_pred_xgb)
precision_xgb = precision_score(y_test_encoded, y_pred_xgb, average='weighted')
recall_xgb = recall_score(y_test_encoded, y_pred_xgb, average='weighted')
f1_xgb = f1_score(y_test_encoded, y_pred_xgb, average='weighted')

In [ ]:
best_nn = neural_netw(X_train_target_genes, y_train)
y_pred_nn = best_nn.predict(X_test_target_genes)

accuracy_nn = accuracy_score(y_test, y_pred_nn)
precision_nn = precision_score(y_test, y_pred_nn, average='weighted')
recall_nn = recall_score(y_test, y_pred_nn, average='weighted')
f1_nn = f1_score(y_test, y_pred_nn, average='weighted')

In [ ]:
classifiers = [
    ("Random Forest", accuracy_rf, precision_rf, recall_rf, f1_rf),
    ("Logistic Regression", accuracy_lr, precision_lr, recall_lr, f1_lr),
    ("SVM", accuracy_svc, precision_svc, recall_svc, f1_svc),
    ("K-Nearest Neighbors", accuracy_knn, precision_knn, recall_knn, f1_knn),
    ("XGBoost", accuracy_xgb, precision_xgb, recall_xgb, f1_xgb),
    ("Neural Network (MLP)", accuracy_nn, precision_nn, recall_nn, f1_nn)
]

for clf_name, accuracy, precision, recall, f1 in classifiers:
    print(f"{clf_name} Performance:")
    print(f"  Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1 Score: {f1:.4f}\n")

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_svc = confusion_matrix(y_test, y_pred_svc)
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_xgb = confusion_matrix(y_test_encoded, y_pred_xgb)
cm_nn = confusion_matrix(y_test, y_pred_nn)

In [ ]:
classifier_names = [
    'Random Forest', 'Logistic Regression', 'SVM',
    'K-Nearest Neighbors', 'XGBoost', 'Neural Network'
]

fig, axes = plt.subplots(2, 3, figsize=(10, 8))
fig.suptitle('Confusion Matrices', fontsize=16)

axes = axes.ravel()


for i, (cm, name) in enumerate(zip(
    [cm_rf, cm_lr, cm_svc, cm_knn, cm_xgb, cm_nn],
    classifier_names
)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

Overall the best F1 score is achieved by XGBoost with parameters {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}.

Next, we evaluate performance using the dataset that uses the full gene subset as features.

In [ ]:
best_rf = random_forest(X_train_subset, y_train)
y_pred_rf = best_rf.predict(X_test_subset)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, average='weighted')
recall_rf = recall_score(y_test, y_pred_rf, average='weighted')
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

In [ ]:
best_lr = logistic_regression(X_train_subset, y_train)
y_pred_lr = best_lr.predict(X_test_subset)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

In [ ]:
best_svc = svm(X_train_subset, y_train)
y_pred_svc = best_svc.predict(X_test_subset)

accuracy_svc = accuracy_score(y_test, y_pred_svc)
precision_svc = precision_score(y_test, y_pred_svc, average='weighted')
recall_svc = recall_score(y_test, y_pred_svc, average='weighted')
f1_svc = f1_score(y_test, y_pred_svc, average='weighted')

In [ ]:
best_lr = logistic_regression(X_train_subset, y_train)
y_pred_lr = best_lr.predict(X_test_subset)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

In [ ]:
best_knn = knn(X_train_subset, y_train)
y_pred_knn = best_knn.predict(X_test_subset)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')

In [ ]:
best_xgb = xgboost(X_train_subset, y_train_encoded)
y_pred_xgb = best_xgb.predict(X_test_subset)

accuracy_xgb = accuracy_score(y_test_encoded, y_pred_xgb)
precision_xgb = precision_score(y_test_encoded, y_pred_xgb, average='weighted')
recall_xgb = recall_score(y_test_encoded, y_pred_xgb, average='weighted')
f1_xgb = f1_score(y_test_encoded, y_pred_xgb, average='weighted')

In [ ]:
best_nn = neural_netw(X_train_subset, y_train)
y_pred_nn = best_nn.predict(X_test_subset)

accuracy_nn = accuracy_score(y_test, y_pred_nn)
precision_nn = precision_score(y_test, y_pred_nn, average='weighted')
recall_nn = recall_score(y_test, y_pred_nn, average='weighted')
f1_nn = f1_score(y_test, y_pred_nn, average='weighted')

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_svc = confusion_matrix(y_test, y_pred_svc)
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_xgb = confusion_matrix(y_test_encoded, y_pred_xgb)
cm_nn = confusion_matrix(y_test, y_pred_nn)

In [ ]:
classifier_names = [
    'Random Forest', 'Logistic Regression', 'SVM',
    'K-Nearest Neighbors', 'XGBoost', 'Neural Network'
]

fig, axes = plt.subplots(2, 3, figsize=(10, 8))
fig.suptitle('Confusion Matrices', fontsize=16)

axes = axes.ravel()


for i, (cm, name) in enumerate(zip(
    [cm_rf, cm_lr, cm_svc, cm_knn, cm_xgb, cm_nn],
    classifier_names
)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In this second case, the top F1 scores are lower than those achieved by the previously best-performing SVM model. As before, the best overall performance is obtained when using the dataset that includes only the subset of target genes as features, employing XGBoost with parameters {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}.

The overall performance of this model is:

Accuracy: 0.8258 | Precision: 0.7895 | Recall: 0.8258 | F1 Score: 0.8065